In [2]:
from pathlib import Path
from typing import Optional, Tuple, Dict, Any

import numpy as np
import tifffile
from liffile import LifFile
from skimage.filters import sobel


In [3]:
def _score_patch_across_z(stack_zyx: np.ndarray, patch_slice: Tuple[slice, slice], focus_mode: str,) -> np.ndarray:
    """Return per-z score vector for one XY patch."""
    mode = str(focus_mode).strip().lower()

    if mode == "brightfield":
        # Best-focus = highest local sharpness (Sobel std).
        return np.array(
            [
                float(np.std(sobel(np.asarray(stack_zyx[z][patch_slice], dtype=np.float32))))
                for z in range(stack_zyx.shape[0])
            ],
            dtype=np.float32,
        )

    if mode == "fluorescence":
        # Best-focus = maximal integrated fluorescence signal in patch.
        return np.array(
            [float(np.sum(np.asarray(stack_zyx[z][patch_slice], dtype=np.float32))) for z in range(stack_zyx.shape[0])],
            dtype=np.float32,
        )

    raise ValueError("focus_mode must be 'brightfield' or 'fluorescence'")

def curved_plane_refocus(
    stack_zyx: np.ndarray,
    grid: int = 10,
    patch: int = 50,
    mask: Optional[np.ndarray] = None,
    focus_mode: str = "brightfield",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Estimate curved best-focus plane from a z-stack.

    Parameters
    ----------
    stack_zyx : np.ndarray
        Focus channel stack in ZYX order.
    grid : int
        Number of sample points per axis for local focus voting.
    patch : int
        Patch size around each sampled point.
    mask : Optional[np.ndarray]
        Optional XY mask to restrict sampled points.
    focus_mode : str
        'brightfield' -> highest Sobel sharpness in each patch.
        'fluorescence' -> highest summed intensity in each patch.

    Returns
    -------
    focus_plane_xy : np.ndarray
        In-focus plane extracted using per-pixel z-map.
    zmap_xy : np.ndarray
        Integer z-index map (shape YX).
    sample_points_xy : np.ndarray
        Sample point coordinates used for fitting.
    sample_best_z : np.ndarray
        Best-z estimate at each sampled point.
    """
    stack_zyx = np.asarray(stack_zyx)
    if stack_zyx.ndim != 3:
        raise ValueError(f"Expected ZYX stack for focus scoring, got shape={stack_zyx.shape}")

    Z, H, W = stack_zyx.shape
    if Z == 0:
        raise ValueError("Empty stack: Z dimension is zero")

    mode = str(focus_mode).strip().lower()
    if mode not in ("brightfield", "fluorescence"):
        raise ValueError("focus_mode must be 'brightfield' or 'fluorescence'")

    if Z == 1:
        zmap = np.zeros((H, W), dtype=np.int16)
        return stack_zyx[0], zmap, np.empty((0, 2), dtype=np.float32), np.empty((0,), dtype=np.float32)

    patch = int(max(5, patch))
    patch = patch if patch % 2 == 0 else patch + 1
    grid = int(max(4, grid))

    ys = np.linspace(patch // 2, H - patch // 2 - 1, grid).astype(int)
    xs = np.linspace(patch // 2, W - patch // 2 - 1, grid).astype(int)

    pts = []
    zs = []
    for y in ys:
        for x in xs:
            if mask is not None and not bool(mask[y, x]):
                continue
            sl = (slice(y - patch // 2, y + patch // 2), slice(x - patch // 2, x + patch // 2))
            f = _score_patch_across_z(stack_zyx, sl, mode)
            pts.append((x, y))
            zs.append(int(np.argmax(f)))

    if len(zs) < 6:
        if mode == "brightfield":
            scores = [float(np.std(sobel(stack_zyx[z].astype(np.float32)))) for z in range(Z)]
        else:
            scores = [float(np.sum(stack_zyx[z].astype(np.float32))) for z in range(Z)]
        z0 = int(np.argmax(scores))
        zmap = np.full((H, W), z0, dtype=np.int16)
        return stack_zyx[z0], zmap, np.asarray(pts, dtype=np.float32), np.asarray(zs, dtype=np.float32)

    pts = np.asarray(pts, dtype=np.float32)
    zs = np.asarray(zs, dtype=np.float32)

    Xc = pts[:, 0] - pts[:, 0].mean()
    Yc = pts[:, 1] - pts[:, 1].mean()
    scale = float(max(W, H))
    Xn = Xc / scale
    Yn = Yc / scale

    B = np.column_stack((Xn**2, Yn**2, Xn * Yn, Xn, Yn, np.ones_like(Xn)))
    coeffs, *_ = np.linalg.lstsq(B, zs, rcond=None)

    Xg, Yg = np.meshgrid(np.arange(W), np.arange(H))
    mean_x = float(pts[:, 0].mean())
    mean_y = float(pts[:, 1].mean())
    Xg_n = (Xg - mean_x) / scale
    Yg_n = (Yg - mean_y) / scale

    zmap = (
        coeffs[0] * Xg_n**2
        + coeffs[1] * Yg_n**2
        + coeffs[2] * Xg_n * Yg_n
        + coeffs[3] * Xg_n
        + coeffs[4] * Yg_n
        + coeffs[5]
    )
    zmap = np.clip(np.rint(zmap).astype(np.int16), 0, Z - 1)

    focus_plane = np.take_along_axis(stack_zyx, zmap[None, :, :], axis=0)[0]
    return focus_plane, zmap, pts, zs

def refocus_stack_around_plane(stack: np.ndarray, zmap_xy: np.ndarray) -> np.ndarray:
    """
    Retilt stack so the curved focus plane becomes the same z-index across XY.

    Supports stack shaped ZYX or ZYXC.
    """
    arr = np.asarray(stack)
    if arr.ndim not in (3, 4):
        raise ValueError(f"Expected stack with ndim 3 or 4, got shape={arr.shape}")

    Z = arr.shape[0]
    H, W = arr.shape[1], arr.shape[2]

    zmap = np.asarray(zmap_xy, dtype=np.int32)
    if zmap.shape != (H, W):
        raise ValueError(f"zmap shape {zmap.shape} does not match stack XY {(H, W)}")

    z_ref = int(np.clip(np.rint(np.median(zmap)), 0, Z - 1))
    offsets = (np.arange(Z, dtype=np.int32) - z_ref)[:, None, None]
    src = np.clip(zmap[None, :, :] + offsets, 0, Z - 1)

    if arr.ndim == 3:
        return np.take_along_axis(arr, src, axis=0)

    out = np.empty_like(arr)
    for c in range(arr.shape[-1]):
        out[..., c] = np.take_along_axis(arr[..., c], src, axis=0)
    return out

def normalize_to_zyxc(arr: np.ndarray, dims: Optional[Tuple[str, ...]] = None) -> np.ndarray:
    """
    Convert many common microscopy array layouts to ZYXC.

    If `dims` is provided (e.g., from xarray dims), it is used directly.
    Otherwise a heuristic path is used.
    """
    arr = np.asarray(arr)

    if dims is not None:
        dims_u = tuple(str(d).upper() for d in dims)
        keep = [i for i, d in enumerate(dims_u) if d in ("Z", "Y", "X", "C")]
        if not keep:
            raise ValueError(f"Could not find Z/Y/X/C dims in provided dims={dims}")
        arr = np.transpose(arr, axes=keep)
        dims_kept = [dims_u[i] for i in keep]

        for axis_name in ("Z", "Y", "X", "C"):
            if axis_name not in dims_kept:
                arr = np.expand_dims(arr, axis=0)
                dims_kept = [axis_name] + dims_kept

        perm = [dims_kept.index("Z"), dims_kept.index("Y"), dims_kept.index("X"), dims_kept.index("C")]
        return np.transpose(arr, axes=perm)

    if arr.ndim == 2:
        return arr[None, :, :, None]

    if arr.ndim == 3:
        # Use channel heuristic first: C <= 4. For 3D arrays this implies single-Z with channel present.
        if arr.shape[-1] <= 4 and arr.shape[0] > 8 and arr.shape[1] > 8:
            # YXC
            return arr[None, :, :, :]
        if arr.shape[0] <= 4 and arr.shape[1] > 8 and arr.shape[2] > 8:
            # CYX
            return np.transpose(arr, (1, 2, 0))[None, :, :, :]
        # Otherwise assume ZYX (single channel).
        return arr[:, :, :, None]

    if arr.ndim == 4:
        s0, s1, s2, s3 = arr.shape

        # Prefer layouts where channel axis satisfies C <= 4 and z axis is larger than C.
        if s3 <= 4 and s0 > s3:
            # ZYXC
            return arr
        if s0 <= 4 and s1 > s0 and s2 > 8 and s3 > 8:
            # CZYX -> ZYXC
            return np.transpose(arr, (1, 2, 3, 0))
        if s1 <= 4 and s0 > s1 and s2 > 8 and s3 > 8:
            # ZCYX -> ZYXC
            return np.transpose(arr, (0, 2, 3, 1))

        # Fallbacks for common cases that may violate strict assumptions.
        if s3 <= 4:
            return arr
        if s0 <= 4 and s2 > 8 and s3 > 8:
            return np.transpose(arr, (1, 2, 3, 0))
        if s1 <= 4 and s2 > 8 and s3 > 8:
            return np.transpose(arr, (0, 2, 3, 1))

    raise ValueError(f"Could not infer array layout for shape={arr.shape}; please reshape to ZYXC before calling")

def _load_source_to_zyxc(
    source_path: str,
    image_index: int = 0,
) -> Tuple[np.ndarray, str]:
    """
    Load either a .lif image (by index) or .tif/.tiff and normalize to ZYXC.
    Returns (stack_zyxc, source_kind).
    """
    p = Path(source_path)
    if not p.exists():
        raise FileNotFoundError(f"Path does not exist: {p}")

    suffix = p.suffix.lower()
    if suffix == ".lif":
        with LifFile(p) as lif:
            idx = int(image_index)
            if idx < 0 or idx >= len(lif.images):
                raise IndexError(f"image_index={idx} is out of range [0, {len(lif.images)-1}]")
            img = lif.images[idx]

            dims = None
            arr = np.asarray(img.asarray())
            try:
                xa = img.asxarray()
                dims = tuple(getattr(xa, "dims", ()))
                arr = np.asarray(xa.values)
            except Exception:
                pass

        return normalize_to_zyxc(arr, dims=dims), "lif"

    if suffix in (".tif", ".tiff"):
        arr = np.asarray(tifffile.imread(str(p)))
        return normalize_to_zyxc(arr, dims=None), "tiff"

    raise ValueError("Unsupported input: use a .lif, .tif, or .tiff path")

def retilt_image(
    source_path: str,
    image_index: int = 0,
    focus_channel: int = 0,
    focus_mode: str = "brightfield",
    n_sampling: int = 10,
    patch: int = 50,
) -> Dict[str, Any]:
    """
    Retilt an image stack around a curved best-focus plane.

    Parameters
    ----------
    source_path : str
        Path to either .lif or .tif/.tiff.
    image_index : int
        Used only for .lif, selects the image within the file.
    focus_channel : int
        Channel index used to estimate the focus plane.
        The computed z-map is then applied to all channels.
    focus_mode : str
        'brightfield' uses patch sharpness (Sobel std).
        'fluorescence' uses patch summed intensity.
    n_sampling : int
        Grid density for focus sampling.
    patch : int
        Patch size for focus scoring.

    Returns
    -------
    dict with keys:
        - retilted_czyx: np.ndarray
        - retilted_zyxc: np.ndarray
        - input_czyx: np.ndarray
        - focus_plane_xy: np.ndarray
        - zmap_xy: np.ndarray
        - input_zyxc: np.ndarray
        - source_kind: str
        - focus_channel: int
        - focus_mode: str
    """
    stack_zyxc, source_kind = _load_source_to_zyxc(source_path=source_path, image_index=image_index)

    Z, Y, X, C = stack_zyxc.shape
    if not (0 <= int(focus_channel) < C):
        raise ValueError(f"focus_channel={focus_channel} is out of range [0, {C-1}]")

    mode = str(focus_mode).strip().lower()
    if mode not in ("brightfield", "fluorescence"):
        raise ValueError("focus_mode must be 'brightfield' or 'fluorescence'")

    if Z == 1:
        zmap = np.zeros((Y, X), dtype=np.int16)
        return {
            "retilted_czyx": np.transpose(stack_zyxc.copy(), (3, 0, 1, 2)),
            "retilted_zyxc": stack_zyxc.copy(),
            "input_czyx": np.transpose(stack_zyxc.copy(), (3, 0, 1, 2)),
            "focus_plane_xy": stack_zyxc[0, :, :, int(focus_channel)],
            "zmap_xy": zmap,
            "input_zyxc": stack_zyxc,
            "source_kind": source_kind,
            "focus_channel": int(focus_channel),
            "focus_mode": mode,
        }

    focus_stack_zyx = stack_zyxc[:, :, :, int(focus_channel)]
    focus_plane_xy, zmap_xy, sample_points_xy, sample_best_z = curved_plane_refocus(
        focus_stack_zyx,
        grid=int(n_sampling),
        patch=int(patch),
        mask=None,
        focus_mode=mode,
    )
    retilted_zyxc = refocus_stack_around_plane(stack_zyxc, zmap_xy)
    retilted_czyx = np.transpose(retilted_zyxc, (3, 0, 1, 2))

    return {
        "retilted_czyx": retilted_czyx,
        "retilted_zyxc": retilted_zyxc,
        "input_czyx": np.transpose(stack_zyxc, (3, 0, 1, 2)),
        "focus_plane_xy": focus_plane_xy,
        "zmap_xy": zmap_xy,
        "input_zyxc": stack_zyxc,
        "source_kind": source_kind,
        "focus_channel": int(focus_channel),
        "focus_mode": mode,
        "sample_points_xy": sample_points_xy,
        "sample_best_z": sample_best_z,
    }


In [4]:
def _safe_name(name: str) -> str:
    """Make a filesystem-safe stem for output files."""
    return "".join(ch if ch.isalnum() or ch in ("-", "_", ".") else "_" for ch in str(name)).strip("_") or "image"

def _iter_retilt_jobs(input_path: str):
    """
    Yield jobs for a single tif/tiff, a single lif (all images), or a folder containing them.
    Each yielded job has: source_path, image_index, output_stem.
    """
    p = Path(input_path)
    if not p.exists():
        raise FileNotFoundError(f"Path does not exist: {p}")

    if p.is_file():
        suffix = p.suffix.lower()
        if suffix in (".tif", ".tiff"):
            yield {"source_path": str(p), "image_index": 0, "output_stem": p.stem}
            return
        if suffix == ".lif":
            with LifFile(p) as lif:
                for idx, img in enumerate(lif.images):
                    img_name = _safe_name(getattr(img, "name", "") or f"image_{idx:03d}")
                    yield {
                        "source_path": str(p),
                        "image_index": idx,
                        "output_stem": f"{p.stem}_{img_name}",
                    }
            return
        raise ValueError("Unsupported input file. Use .lif, .tif, .tiff, or a folder.")

    if p.is_dir():
        image_files = sorted(
            [q for q in p.iterdir() if q.is_file() and q.suffix.lower() in (".lif", ".tif", ".tiff")],
            key=lambda q: q.name.lower(),
        )
        if not image_files:
            raise ValueError(f"No .lif/.tif/.tiff files found in folder: {p}")

        for q in image_files:
            if q.suffix.lower() in (".tif", ".tiff"):
                yield {"source_path": str(q), "image_index": 0, "output_stem": q.stem}
            else:
                with LifFile(q) as lif:
                    for idx, img in enumerate(lif.images):
                        img_name = _safe_name(getattr(img, "name", "") or f"image_{idx:03d}")
                        yield {
                            "source_path": str(q),
                            "image_index": idx,
                            "output_stem": f"{q.stem}_{img_name}",
                        }
        return

    raise ValueError(f"Unsupported path type: {p}")

def retilt_and_save(
    input_path: str,
    output_dir: str,
    focus_channel: int = 0,
    focus_mode: str = "brightfield",
    n_sampling: int = 10,
    patch: int = 50,
    overwrite: bool = False,
) -> Dict[str, Any]:
    """
    Batch retilt stacks from a tif/tiff, a lif, or a folder and save all outputs as TIFF.

    Output naming: <original_name>_retilted.tif
    - tif/tiff: original_name = file stem
    - lif images: original_name = <lif_stem>_<lif_image_name>

    Saved TIFF axis order for ImageJ compatibility: ZCYX
    
    Returns a summary dict with saved and failed jobs.
    """
    mode = str(focus_mode).strip().lower()
    if mode not in ("brightfield", "fluorescence"):
        raise ValueError("focus_mode must be 'brightfield' or 'fluorescence'")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    failed = []

    for job in _iter_retilt_jobs(input_path):
        src = job["source_path"]
        idx = int(job["image_index"])
        stem = _safe_name(job["output_stem"])
        out_path = out_dir / f"{stem}_retilted.tif"

        if out_path.exists() and not overwrite:
            failed.append({
                "source": src,
                "image_index": idx,
                "output": str(out_path),
                "error": "Output exists (set overwrite=True to replace)",
            })
            continue

        try:
            result = retilt_image(
                source_path=src,
                image_index=idx,
                focus_channel=int(focus_channel),
                focus_mode=mode,
                n_sampling=int(n_sampling),
                patch=int(patch),
            )
            # Internal output is ZYXC; write as ZCYX for ImageJ hyperstack compatibility.
            retilted_zcyx = np.transpose(np.asarray(result["retilted_zyxc"]), (0, 3, 1, 2))
            tifffile.imwrite(
                str(out_path),
                retilted_zcyx,
                imagej=True,
                metadata={"axes": "ZCYX", "hyperstack": True},
            )
            saved.append({
                "source": src,
                "image_index": idx,
                "output": str(out_path),
                "shape_zcyx": tuple(retilted_zcyx.shape),
            })
        except Exception as exc:
            try:
                if out_path.exists():
                    out_path.unlink()
            except Exception:
                pass
            failed.append({
                "source": src,
                "image_index": idx,
                "output": str(out_path),
                "error": str(exc),
            })

    return {
        "input_path": str(Path(input_path)),
        "output_dir": str(out_dir),
        "focus_mode": mode,
        "focus_channel": int(focus_channel),
        "saved": saved,
        "failed": failed,
        "n_saved": len(saved),
        "n_failed": len(failed),
    }

In [5]:
# Batch usage: input can be a .lif, .tif/.tiff, or a folder containing these.
input_path = r"Z:\Bel\Marina_Retilting\Inputs"
output_dir = r"Z:\Bel\Marina_Retilting\Outputs"

# Choose exactly one mode: "fluorescence" or "brightfield"
focus_mode = "fluorescence"

batch_result = retilt_and_save(
    input_path=input_path,
    output_dir=output_dir,
    focus_channel=0,
    focus_mode=focus_mode,
    n_sampling=12,
    patch=48,
    overwrite=True,
 )

batch_result["n_saved"], batch_result["n_failed"]

(7, 0)

In [ ]:
test = tifffile.imread("z:\Bel\Marina_Retilting\Outputs\Vessel_formation_Fig1_small_Day2_HUVECmCherry_Merged_retilted.tif")
test.shape

(2, 38, 2823, 3284)

In [ ]:
# Optional: inspect what was written
batch_result["saved"][:3], batch_result["failed"][:3]

([],
 [{'source': 'Z:\\Bel\\Marina_Retilting\\Inputs\\marina_retilt_eg.tif',
   'image_index': 0,
   'output': 'Z:\\Bel\\Marina_Retilting\\Outputs\\marina_retilt_eg_retilted.tif',
   'error': 'Output exists (set overwrite=True to replace)'},
  {'source': 'Z:\\Bel\\Marina_Retilting\\Inputs\\marina_retilt_eg2.tif',
   'image_index': 0,
   'output': 'Z:\\Bel\\Marina_Retilting\\Outputs\\marina_retilt_eg2_retilted.tif',
   'error': 'Output exists (set overwrite=True to replace)'},
  {'source': 'Z:\\Bel\\Marina_Retilting\\Inputs\\Vessel formation_Fig1_small.lif',
   'image_index': 0,
   'output': 'Z:\\Bel\\Marina_Retilting\\Outputs\\Vessel_formation_Fig1_small_Day2_HUVECmCherry_Merged_retilted.tif',
   'error': 'Output exists (set overwrite=True to replace)'}])

In [ ]:
import napari
viewer = napari.Viewer()
viewer.add


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


AttributeError: 'Viewer' object has no attribute 'add'

In [ ]:
out_bf["retilted_czyx"].shape

(2, 34, 3285, 3278)

In [6]:
# Diagnostics: check save failures and on-disk file sizes
from pathlib import Path

print('n_saved:', batch_result.get('n_saved'))
print('n_failed:', batch_result.get('n_failed'))
print('failed_sample:', batch_result.get('failed', [])[:3])

saved = batch_result.get('saved', [])
if saved:
    for item in saved[:5]:
        p = Path(item['output'])
        print(p.name, 'exists=', p.exists(), 'size_bytes=', p.stat().st_size if p.exists() else None)

n_saved: 7
n_failed: 0
failed_sample: []
marina_retilt_eg_retilted.tif exists= True size_bytes= 732251114
marina_retilt_eg2_retilted.tif exists= True size_bytes= 704588434
Vessel_formation_Fig1_small_Day2_HUVECmCherry_Merged_retilted.tif exists= True size_bytes= 704588434
Vessel_formation_Fig1_small_Day4_HUVECmCherry_Merged_retilted.tif exists= True size_bytes= 732251114
Vessel_formation_Fig1_small_Day6_HUVECmCherry_Merged_retilted.tif exists= True size_bytes= 612065772
